# Explainable Customer Churn Prediction and Retention Strategy
## Model Comparison: Logistic Regression, Random Forest, and XGBoost

## 1. Import Libraries
## 2. Load Data
## 3. Data Cleaning and Preparation
## 4. Exploratory Data Analysis
## 5. Feature Engineering
## 6. Train-Test Split
## 7. Logistic Regression
## 8. Random Forest
## 9. XGBoost
## 10. Model Comparison
## 11. Threshold Optimization
## 12. Explainability
## 13. Retention Strategy
## 14. Export Outputs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

print("All libraries imported successfully")

In [ ]:
df = pd.read_excel("../data/Telco_churn.xlsx")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df_clean.head()

In [ ]:
df_clean.to_csv("../outputs/data/df_clean.csv", index=False)

In [ ]:
df_clean.shape

In [ ]:
df_clean['Churn Value'] = df_clean['Churn Value'].astype(int)

In [ ]:
df_clean = df_clean.drop(columns=[
    'Churn Label',
    'Churn Score',
    'Churn Reason'
], errors='ignore')

In [ ]:
df_clean.shape

In [ ]:
# Drop irrelevant columns
cols_to_drop = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude'
]

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
df_clean.shape

In [ ]:
df_clean.to_csv("../outputs/data/df_clean.csv", index=False)
X = df_clean.drop('Churn Value', axis=1)
y = df_clean['Churn Value']

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns
categorical_cols

In [ ]:
df_clean = df.copy()
cols_to_drop = [
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Score', 'Churn Reason'
]

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')
df_clean = df_clean.dropna()
X = df_clean.drop('Churn Value', axis=1)
y = df_clean['Churn Value']
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_encoded.shape



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# Check Split
X_train.shape, X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
# Save Split data
X_train.to_csv("../outputs/data/X_train.csv", index=False)
X_test.to_csv("../outputs/data/X_test.csv", index=False)
y_train.to_csv("../outputs/data/y_train.csv", index=False)
y_test.to_csv("../outputs/data/y_test.csv", index=False)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

lr.fit(X_train_scaled, y_train)

In [ ]:
# make predixtion
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

In [ ]:
lr_auc = roc_auc_score(y_test, y_proba_lr)

## LR Interpretation
Good at detecting churners (high recall)
But has false positives (lower precision)


In [ ]:
## Train baseline Random Forest

rf.fit(X_train, y_train)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

In [ ]:
## Evaluate RF

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

In [ ]:
## Parameter Tuning RF
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)


In [ ]:
## Best Param tuning RF
best_rf = grid_search.best_estimator_
print(grid_search.best_params_)

In [ ]:
## Evaluate Tuned RF
y_pred_rf_best = best_rf.predict(X_test)
y_proba_rf_best = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf_best))

In [ ]:
#'Train Baseline XGBoost
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

In [ ]:
## Prediction
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

## Evaluate
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
## XGBoost Tuning(Define the parameter grid)
from sklearn.model_selection import GridSearchCV

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

In [ ]:
grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

In [ ]:
best_xgb = grid_xgb.best_estimator_
print(grid_xgb.best_params_)

In [ ]:
## Evaluate Tuned Parameter XGBOOST
y_pred_xgb_best = best_xgb.predict(X_test)
y_proba_xgb_best = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_xgb_best))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb_best))

## Business Decision
Although XGBoost achieved a marginally higher ROC-AUC, Random Forest was selected as the final model due to its superior balance between recall and precision. 

In a churn prediction context, missing potential churners carries a higher business cost than false positives. Random Forest achieved a significantly higher recall (0.77 vs 0.56), ensuring more at-risk customers are identified for proactive retention strategies, while maintaining a strong F1 score for overall model effectiveness.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance.head()

In [ ]:
feature_importance.to_csv("../outputs/features/feature_importance.csv", index=False)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []

for t in thresholds:
    y_pred_t = (y_proba_rf_best >= t).astype(int)
    
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    
    threshold_results.append({
        'Threshold': t,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

## Choose a Threshold
A threshold of 0.50 was selected as it provides the optimal balance between precision and recall. This ensures that a significant proportion of at-risk customers are identified for intervention while maintaining manageable false positive rates, supporting efficient allocation of retention resources.

In [ ]:
## Creating a Risk Segmentation(High,medium and Low)
df_strategy = X_test.copy()

df_strategy['Actual'] = y_test
df_strategy['Churn_Probability'] = y_proba_rf_best
def assign_risk(p):
    if p >= 0.6:
        return 'High Risk'
    elif p >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_strategy['Risk_Level'] = df_strategy['Churn_Probability'].apply(assign_risk)

In [ ]:
## Add Business Actions
def action_map(risk):
    if risk == 'High Risk':
        return 'Immediate retention offer'
    elif risk == 'Medium Risk':
        return 'Engagement campaign'
    else:
        return 'Maintain relationship'

df_strategy['Recommended_Action'] = df_strategy['Risk_Level'].apply(action_map)

In [ ]:
df_strategy[['Churn_Probability','Risk_Level','Recommended_Action']].head()

In [ ]:
df_strategy.to_csv("../outputs/data/df_strategy.csv", index=False)

In [ ]:
import shap

In [ ]:
##Using BEST Model Ramdom Forest)
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

In [ ]:
import numpy as np

print(type(shap_values))
print(np.array(shap_values).shape)
print(X_test.shape)

In [ ]:
shap_values_churn = shap_values[:, :, 1]

In [ ]:
shap.summary_plot(shap_values_churn, X_test)

In [ ]:
## Bar plot version
shap.summary_plot(shap_values_churn, X_test, plot_type="bar")

In [ ]:
import matplotlib.pyplot as plt

shap.summary_plot(shap_values_churn, X_test, show=False)
plt.savefig("../outputs/figures/shap_summary.png", bbox_inches='tight')
plt.close()

In [ ]:
import joblib

joblib.dump(best_rf, "../models/random_forest_final.pkl")

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision': [0.50, 0.54, 0.64],
    'Recall': [0.79, 0.77, 0.56],
    'F1 Score': [0.61, 0.64, 0.60],
    'ROC-AUC': [0.8426, 0.8446, 0.8486]
})

In [ ]:
comparison.to_csv("../outputs/metrics/model_comparison.csv", index=False)

In [ ]:
threshold_df.to_csv("../outputs/metrics/threshold_optimization.csv", index=False)

In [ ]:
import numpy as np

shap_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_SHAP_Impact': np.abs(shap_values_churn).mean(axis=0)
}).sort_values(by='Mean_SHAP_Impact', ascending=False)

shap_importance.head()

In [ ]:
shap_importance.to_csv("../outputs/features/shap_feature_importance.csv", index=False)

In [ ]:
df_full = X_encoded.copy()

df_full['Actual'] = y
df_full['Churn_Probability'] = best_rf.predict_proba(X_encoded)[:, 1]

In [ ]:
df_full.to_csv("../outputs/data/powerbi_dataset.csv", index=False)

In [ ]:
df_full = df_clean.copy()
df_full['Churn_Probability'] = best_rf.predict_proba(X_encoded)[:, 1]
def assign_risk(p):
    if p >= 0.6:
        return 'High Risk'
    elif p >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_full['Risk_Level'] = df_full['Churn_Probability'].apply(assign_risk)
def action_map(risk):
    if risk == 'High Risk':
        return 'Immediate retention offer'
    elif risk == 'Medium Risk':
        return 'Engagement campaign'
    else:
        return 'Maintain relationship'

df_full['Recommended_Action'] = df_full['Risk_Level'].apply(action_map)
df_full.to_csv("../outputs/powerbi_dataset.csv", index=False)